# 1 · Raw IQ bridge

**Local Quartus workflow · host or container · one active FPGA design at a time**

`x[n]=I[n]+jQ[n]`

**What is added:** A raw-IQ bridge between three ESP32 UARTs and host Python.

**Why it is useful:** See the received samples and establish a trustworthy input before adding DSP.

**How it works:** FPGA UART parsers accept signed CI8 I/Q pairs, check framing/CRC and freeze a burst in RAM. The host reads an IQB1 frame and draws its FFT.

```text
ESP32 → UART / CRC → FPGA RAM → Python
```

Python model → native VDS diagram → real RTL/IP simulation → SOF → board → focused CyberEther page.

## Setup

Once, from this example folder: `bash scripts/install_local.sh --system` (Linux system packages need sudo), then select **Atum A3 IQ Demos** as the notebook kernel. Quartus/Questa/Nios tools must already be installed. Set the optional paths below if they are not on PATH. No Docker commands are needed. Long-running cells show an animated activity bar, elapsed time and the active step/log; a full bar means successful completion, not an estimated compile percentage.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/lessons.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "cyberether"))
# Optional overrides, set before tool discovery:
# os.environ["QUARTUS_ROOTDIR"] = "/your/install/quartus"
# os.environ["QUESTA_ROOT"] = "/your/install/questa_fse"
# os.environ["ATUM_ALTERA_ROOT"] = "/your/install"
# os.environ["ATUM_UART"] = "/dev/serial/by-id/your-board"
# os.environ["ATUM_JTAG_CABLE"] = "your cable from jtagconfig"
from lessons import Lesson, lan_ip, stop_demo
from common import tool_exe
CHECKPOINT = 1
FORCE_REBUILD = False   # Fresh checkout builds; source-matched SOFs are reused afterwards.
lesson = Lesson(CHECKPOINT)
HOST = lan_ip()
print("Python:", sys.executable)
print("Quartus:", tool_exe("quartus_sh"))
print("Questa:", tool_exe("vsim"))
print("Browser host:", HOST)


## 1 · Python simulation

The code below is the editable numerical model. Its plots are synthetic/theory, separate from the RTL results and live hardware below.

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    import numpy as np
    import matplotlib.pyplot as plt
    from notebook_plots import spectrum
    from nios_fir import PROFILES, fixed_reference
    from nios_ddc import reference, gain_reference
    from iq_lab_samples import load_sample
    raw, _ = load_sample("two-tone")
    fs = 16e6
    n = CHECKPOINT
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    axs[0].plot(np.arange(64) / 16, raw[:64, 0], '.-', label='I')
    axs[0].plot(np.arange(64) / 16, raw[:64, 1], '.-', label='Q')
    axs[0].set(xlabel='Time (µs)', ylabel='Signed CI8 counts', title='Two numbers per complex sample')
    axs[0].legend()
    axs[1].scatter(raw[:256, 0], raw[:256, 1], s=12)
    axs[1].set(xlabel='I', ylabel='Q', title='Raw IQ · no demodulation')
    spectrum(axs[2], raw.astype(float), fs, 'Python model')
    summary = {'sample_format': 'CI8: I,Q bytes', 'burst_samples': 4096, 'nominal_burst_us': 256}
    fig.suptitle(f"{CHECKPOINT} · Python simulation / integer reference")
    fig.tight_layout()
    plt.show()
    summary

summary

## 2 · Generate the actual FPGA block design

Rebuild the Nios initialization memory where present, recreate the native VDS system, validate and generate vendor IP. The cell opens **Quartus Visual Designer Studio itself**, fits the diagram and uses **File → Export**. Its native PDF is rendered to a high-resolution PNG and displayed inline. The original PDF and a full GUI screenshot are saved beside it. A private X display keeps capture separate from your desktop. The output also provides **Quartus project** and **complete project ZIP** links. The ZIP contains the SDC, RTL, generated IP and Nios memory initialization. Extract it before opening the QPF in Quartus.

Timing constraints: [iq_bridge_vds.sdc](../fpga_hub/iq_bridge/vds_demo_01_raw/iq_bridge_vds.sdc).

In [ ]:
from IPython.display import Image, display
diagram_png = lesson.generate()
display(Image(filename=str(diagram_png)))
print("Exported:", diagram_png)

### Saved native VDS GUI diagram

![Checkpoint 1 — actual VDS GUI export](diagrams/01_vds_gui.png)

The cell above refreshes this diagram from your local design.

## 3 · RTL/IP simulation

Real UART TX/RX module loopback, binary bytes and 8N1 framing.

The plots below read the actual simulator output files. Assertions must pass before the result is plotted.

In [ ]:
simulation_result = lesson.simulate()
print("Simulation:", simulation_result["status"])
lesson.simulation_plot()

### Open the Questa GUI

Use a desktop terminal **on the machine running Quartus** (or its container desktop). Execute the printed commands. `wave.wlf` opens the saved real simulation; `view.do` adds its signals. To rerun interactively, start `vsim -gui` in the same directory and type `do gui.do` in the Transcript. This reruns the same assertions and leaves the waveform window open.

In [ ]:
lesson.questa_gui()

## 4 · Build the SOF

A source-matched, hash-verified cached bitfile can be reused for a quick presentation. Set `FORCE_REBUILD=True` to demonstrate a full Quartus compilation. A fresh checkout always requires a build. Both a new build and cached reuse display links to the complete Quartus project and its ZIP. Open the linked QPF (or extract the ZIP first), then use **Processing → Start Compilation** to build in the GUI.

In [ ]:
build = lesson.build(force=FORCE_REBUILD)
print("SOF:", lesson.stage / "iq_bridge_vds.sof")
print("SHA256:", build["sha256"])
print("Compiled resources:", build["resources"])

## 5 · Program the physical board

This cell **programs the selected checkpoint**. It first stops a previous notebook-owned demo and releases UART/JTAG. Stop any unrelated acquisition service before running it; an unknown serial owner is refused. No ESP32 flashing or reset-button presses are needed with the installed suite.

In [ ]:
lesson.program()

## 6 · Focused CyberEther demo

Start with **Software preview** to see a known two-tone input without hardware. Return to hardware and look for **VERIFIED**, fresh CRC/metadata, and advancing capture counts. This raw bridge has no internal test generator; the preview does not verify the FPGA.

The page uses the bitfile just programmed. Diagnostic plots work without WebRTC; the native CyberEther canvas is expandable below them. **VERIFIED** requires a fresh matching result; **APPLYING**, **HELD**, **STALE** or **ERROR** keeps old data visibly separate from successful application. **Software preview** is host-only and never counts as a hardware pass. See [demo guide](../docs/NOTEBOOK_DEMOS.md). One board and one acquisition owner are shared across all notebooks.

In [ ]:
live = lesson.start_demo(host=HOST)
print(live)

Leave the page running while presenting. The next notebook stops it before programming. To stop it manually, run `stop_demo()` in the cell below. Live invites and captures stay in ignored runtime folders; do not publish their tokens.

In [ ]:
# stop_demo()